In [1]:
import numpy as np
import pandas as pd

In [ ]:
factor = pd.read_parquet('C:/Users/User/OneDrive - CUHK-Shenzhen/data/factor/sample_oot.parquet')
ind = pd.read_csv('C:/Users/User/OneDrive - CUHK-Shenzhen/data/industry.csv')
factor = factor.merge(ind, on='ts_code', how='left')f
#daily_corr = factor.groupby('trade_date').apply(lambda x: x[factor.columns[8:]].corrwith(x['Y_20d'])).reset_index()

In [17]:
factor

,trade_date,ts_code,next_open,adj_factor,20d_returns,bench_20d_return,20d_excess_returns,Y_20d,daily_std,cum_range,...,l1_code,l1_name,l2_code,l2_name,l3_code,l3_name,name,in_date,out_date,is_new
0,2023-01-03,000001.SZ,1562.065302,113.9362,0.038658,0.078447,-0.039789,0.296183,-0.664492,0.007231,...,801780.SI,银行,801783.SI,股份制银行Ⅱ,857831.SI,股份制银行Ⅲ,平安银行,19910403,NaN,Y
1,2023-01-03,000002.SZ,3154.030700,172.8236,-0.038904,0.078447,-0.117351,0.076750,-0.060682,-0.395396,...,801180.SI,房地产,801181.SI,房地产开发,851811.SI,住宅开发,万科A,19910129,NaN,Y
2,2023-01-03,000004.SZ,40.192960,4.0640,0.054601,0.078447,-0.023847,0.391508,-0.057463,2.138654,...,801750.SI,计算机,801104.SI,软件开发,851042.SI,横向通用软件,*ST国华,20210730,NaN,Y
3,2023-01-03,000005.SZ,17.238480,9.2680,-0.016129,0.078447,-0.094576,0.112472,-1.132833,-1.021928,...,801970.SI,环保,801971.SI,环境治理,859714.SI,综合环境治理,ST星源(退市),20170629,NaN,Y
4,2023-01-03,000006.SZ,238.312800,38.9400,-0.119281,0.078447,-0.197728,0.020208,1.522693,0.442603,...,801180.SI,房地产,801181.SI,房地产开发,851811.SI,住宅开发,深振业A,20150930,NaN,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3980172,2026-03-31,688809.SH,315.000000,1.0000,0.293270,0.074327,0.218943,0.940561,2.988359,-1.662865,...,801080.SI,电子,801081.SI,半导体,850818.SI,半导体设备,强一股份,20251212,NaN,Y
3980173,2026-03-31,688819.SH,35.773890,1.0930,-0.004583,0.074327,-0.078910,0.404647,-0.843225,-0.518119,...,801730.SI,电力设备,801737.SI,电池,857375.SI,蓄电池及其他电池,天能股份,20210104,NaN,Y
3980174,2026-03-31,688981.SH,96.400000,1.0000,0.175415,0.074327,0.101088,0.870087,-0.532745,-0.166347,...,801080.SI,电子,801081.SI,半导体,850816.SI,集成电路制造,中芯国际,20200706,NaN,Y
3980175,2026-03-31,689009.SH,46.278400,1.0330,-0.017857,0.074327,-0.092184,0.340174,-0.806683,-0.682829,...,801880.SI,汽车,801881.SI,摩托车及其他,858811.SI,其他运输设备,九号公司-WD,20201029,NaN,Y


In [ ]:
# 2. 行业+市值中性化
def neutralize_date(group):
    X = pd.concat([group['lncap'], pd.get_dummies(group[industry_col], drop_first=True)], axis=1)
    X = sm.add_constant(X)
    model = sm.OLS(group[factor_col], X, missing='drop')
    res = model.fit()
    group['factor_neutral'] = res.resid
    return group

factor = factor.groupby('trade_date').apply(neutralize_date).reset_index(drop=True)